# Van de Vusse System Identification

Canonical Van de Vusse system-identification notebook. The active workflow solves the benchmark steady state, computes a direct local linearization, discretizes it with ZOH, and uses three nonlinear step tests only for validation and scaling.

## User Config

In [ ]:
import os

from systems.vandevusse import get_vandevusse_notebook_defaults
from utils.notebook_setup import prepare_vandevusse_notebook_env, print_grouped_notebook_summary

NB = get_vandevusse_notebook_defaults("system_identification")
RUN_NEW_EXPERIMENTS = NB["run_new_experiments"]
SHOW_LINEARIZATION_DIAGNOSTICS = NB["show_linearization_diagnostics"]
SHOW_VALIDATION_PLOTS = NB["show_validation_plots"]
SAVE_METADATA_JSON = NB["save_metadata_json"]
VANDEVUSSE_DATA_DIR_OVERRIDE = NB["data_dir_override"]
VANDEVUSSE_RESULTS_DIR_OVERRIDE = NB["results_dir_override"]

REPO_ROOT, DATA_DIR, RESULT_DIR = prepare_vandevusse_notebook_env(
    data_dir_override=VANDEVUSSE_DATA_DIR_OVERRIDE,
    results_dir_override=VANDEVUSSE_RESULTS_DIR_OVERRIDE,
)
os.chdir(REPO_ROOT)

## Imports And Setup

In [ ]:
import numpy as np
import pandas as pd

from systems.vandevusse import (
    VANDEVUSSE_SYSTEM_METADATA,
    apply_vandevusse_deviation_form,
    build_vandevusse_step_test_inputs,
    build_vandevusse_system,
    compute_vandevusse_min_max_states,
    discretize_vandevusse_linear_model,
    linearize_vandevusse_continuous,
    run_vandevusse_step_test_experiment,
    save_vandevusse_identification_artifacts,
    scaling_min_max_factors,
    simulate_vandevusse_system,
    solve_vandevusse_nominal_steady_state,
    validate_vandevusse_linearized_model,
)

np.set_printoptions(precision=6, suppress=True, linewidth=140)

## Benchmark Parameters And Operating Point

In [ ]:
SYS = NB["system_setup"]
LINEARIZATION_CFG = NB["linearization"]
delta_t = float(SYS["delta_t_hours"])
system_params = np.asarray(SYS["system_params"], float).copy()
design_params = np.asarray(SYS["design_params"], float).copy()
ss_inputs = np.asarray(SYS["ss_inputs"], float).copy()
benchmark_state_seed = np.asarray(SYS["benchmark_state_seed"], float).copy()
input_bounds = {
    "u_min": np.asarray(SYS["input_bounds"]["u_min"], float).copy(),
    "u_max": np.asarray(SYS["input_bounds"]["u_max"], float).copy(),
}
step_tests = build_vandevusse_step_test_inputs(
    ss_inputs=ss_inputs,
    step_tests=NB["step_tests"],
    delta_t=delta_t,
    initial_hold_hours=NB["initial_hold_hours"],
    step_hold_hours=NB["step_hold_hours"],
    input_bounds=input_bounds,
)

## Resolved Summary

In [ ]:
print_grouped_notebook_summary(
    "Van de Vusse system-identification summary",
    {
        "Paths": {
            "Repo root": REPO_ROOT,
            "Data dir": DATA_DIR,
            "Results dir": RESULT_DIR,
        },
        "Workflow": {
            "Identification method": "direct_local_linearization",
            "Run new experiments": RUN_NEW_EXPERIMENTS,
            "Show linearization diagnostics": SHOW_LINEARIZATION_DIAGNOSTICS,
            "Show validation plots": SHOW_VALIDATION_PLOTS,
            "Save metadata json": SAVE_METADATA_JSON,
        },
        "Operating point": {
            "Benchmark state seed": benchmark_state_seed,
            "Steady inputs": ss_inputs,
            "Sample time (h)": delta_t,
        },
        "Step tests": {
            test_cfg["name"]: {
                "file": test_cfg["save_filename"],
                "step_delta": test_cfg["step_delta"],
            }
            for test_cfg in step_tests
        },
    },
)

## Instantiate Nonlinear Plant

In [ ]:
plant = build_vandevusse_system(
    params=system_params,
    design_params=design_params,
    ss_inputs=ss_inputs,
    delta_t=delta_t,
    deviation_form=False,
)
plant

## Solve Steady State

In [ ]:
steady_states = solve_vandevusse_nominal_steady_state(
    system_params=system_params,
    design_params=design_params,
    ss_inputs=ss_inputs,
    delta_t=delta_t,
)
steady_states

## Compute Continuous-Time Linearization

In [ ]:
continuous_model = linearize_vandevusse_continuous(
    system_params=system_params,
    design_params=design_params,
    x_ss=steady_states["x_ss"],
    u_ss=steady_states["ss_inputs"],
    linearization_cfg=LINEARIZATION_CFG,
    delta_t=delta_t,
)

if SHOW_LINEARIZATION_DIAGNOSTICS:
    print("State perturbations:", continuous_model["state_eps"])
    print("Input perturbations:", continuous_model["input_eps"])
    print("A_c:\n", continuous_model["A_c"])
    print("B_c:\n", continuous_model["B_c"])
    print("C_c:\n", continuous_model["C_c"])
    print("D_c:\n", continuous_model["D_c"])

## Discretize To A, B, C, D

In [ ]:
discrete_model = discretize_vandevusse_linear_model(
    continuous_model["A_c"],
    continuous_model["B_c"],
    continuous_model["C_c"],
    continuous_model["D_c"],
    delta_t=delta_t,
    method=LINEARIZATION_CFG["discretization_method"],
)
system_dict = {
    "A": discrete_model["A_d"],
    "B": discrete_model["B_d"],
    "C": discrete_model["C_d"],
    "D": discrete_model["D_d"],
}
print({key: value.shape for key, value in system_dict.items()})

## Generate Three Local Validation Step Tests

In [ ]:
for step_cfg in step_tests:
    print(f"{step_cfg['name']}: file={step_cfg['save_filename']}, delta={step_cfg['step_delta']}")

## Simulate Nonlinear Plant Data

In [ ]:
step_results_by_name = {}
absolute_dfs = {}
file_paths = {}

for step_cfg in step_tests:
    csv_path = DATA_DIR / step_cfg["save_filename"]
    if RUN_NEW_EXPERIMENTS:
        result = run_vandevusse_step_test_experiment(
            step_cfg=step_cfg,
            system_params=system_params,
            design_params=design_params,
            ss_inputs=ss_inputs,
            delta_t=delta_t,
            data_dir=DATA_DIR,
            result_dir=RESULT_DIR,
            show_plot=False,
        )
    else:
        if not csv_path.exists():
            raise FileNotFoundError(f"Missing existing step-test file: {csv_path}")
        replay_system = build_vandevusse_system(
            params=system_params,
            design_params=design_params,
            ss_inputs=ss_inputs,
            delta_t=delta_t,
            deviation_form=False,
        )
        result = simulate_vandevusse_system(replay_system, step_cfg["input_sequence"])
        result["csv_path"] = csv_path
        result["time"] = np.arange(result["outputs"].shape[0], dtype=float) * delta_t
    step_results_by_name[step_cfg["name"]] = result
    file_paths[step_cfg["name"]] = result["csv_path"]
    absolute_dfs[step_cfg["name"]] = pd.read_csv(result["csv_path"])

file_paths

## Simulate Linearized Model On The Same Inputs

In [ ]:
deviation_dfs = apply_vandevusse_deviation_form(steady_states, file_paths)
{key: df.head(3) for key, df in deviation_dfs.items()}

## Compare Nonlinear Vs Linearized Responses

In [ ]:
validation = validate_vandevusse_linearized_model(
    system_dict=system_dict,
    absolute_dfs=absolute_dfs,
    deviation_dfs=deviation_dfs,
    step_tests=step_tests,
    steady_states=steady_states,
    delta_t=delta_t,
    result_dir=RESULT_DIR,
    show_plot=SHOW_VALIDATION_PLOTS,
)
validation_metrics = validation["metrics_by_test"]
validation_metrics

## Compute Scaling Artifacts

In [ ]:
data_min, data_max = scaling_min_max_factors(file_paths)
scaling_factor = {
    "min": np.asarray(data_min, float),
    "max": np.asarray(data_max, float),
}
min_max_states = compute_vandevusse_min_max_states(step_results_by_name, steady_states)

print("scaling_factor:")
print(scaling_factor)
print("min_max_states:")
print(min_max_states)

## Save Artifacts And Metadata

In [ ]:
metadata = {
    "identification_method": "direct_local_linearization",
    "system_name": VANDEVUSSE_SYSTEM_METADATA["system_name"],
    "system_params": system_params.copy(),
    "design_params": design_params.copy(),
    "benchmark_state_seed": benchmark_state_seed.copy(),
    "ss_inputs": steady_states["ss_inputs"].copy(),
    "solved_x_ss": steady_states["x_ss"].copy(),
    "solved_y_ss": steady_states["y_ss"].copy(),
    "output_definition": ["c_B", "T"],
    "linearization_cfg": {
        "state_eps_rel": float(LINEARIZATION_CFG["state_eps_rel"]),
        "state_eps_abs": float(LINEARIZATION_CFG["state_eps_abs"]),
        "input_eps_rel": float(LINEARIZATION_CFG["input_eps_rel"]),
        "input_eps_abs": float(LINEARIZATION_CFG["input_eps_abs"]),
        "discretization_method": str(LINEARIZATION_CFG["discretization_method"]),
    },
    "continuous_model": {
        "A_c": continuous_model["A_c"],
        "B_c": continuous_model["B_c"],
        "C_c": continuous_model["C_c"],
        "D_c": continuous_model["D_c"],
        "state_eps": continuous_model["state_eps"],
        "input_eps": continuous_model["input_eps"],
    },
    "discrete_model": {
        "A": system_dict["A"],
        "B": system_dict["B"],
        "C": system_dict["C"],
        "D": system_dict["D"],
    },
    "step_tests": [
        {
            "name": step_cfg["name"],
            "save_filename": step_cfg["save_filename"],
            "input_index": step_cfg["input_index"],
            "step_delta": np.asarray(step_cfg["step_delta"], float).copy(),
            "csv_path": file_paths[step_cfg["name"]],
        }
        for step_cfg in step_tests
    ],
    "validation_metrics": validation_metrics,
    "csv_columns": ["F", "Q_K", "c_B", "T"],
    "run_new_experiments": RUN_NEW_EXPERIMENTS,
}

artifact_paths = save_vandevusse_identification_artifacts(
    repo_root=REPO_ROOT,
    data_dir=DATA_DIR,
    result_dir=RESULT_DIR,
    system_dict=system_dict,
    scaling_factor=scaling_factor,
    min_max_states=min_max_states,
    metadata=metadata,
    save_metadata_json=SAVE_METADATA_JSON,
)
artifact_paths

## Plotting And Export Summary

In [ ]:
print("Saved step-test CSV files:")
for path in sorted(file_paths.values(), key=str):
    print(f"  {path.name}")

print("\nSaved result figures:")
for path in sorted(RESULT_DIR.glob("*.png"), key=str):
    print(f"  {path.name}")

print("\nSaved identification artifacts:")
for label, path in artifact_paths.items():
    print(f"  {label}: {path}")